# 00 — Meeting tour: end-to-end MIDAS v2 demo

**Audience**: anyone you're walking through MIDAS for the first time.
**Time**: ~15 minutes (most cells are seconds; the Hydra real-data
cell is 4–5 min so park it last).
**What it covers**: every Phase 0–6 capability shipped in v0.10.

This notebook is the live-demo flow used in the LANL / Wenqian / APS
management briefings. Each cell runs one demo runner; the markdown
above each cell is the talking point.


## 0. Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import subprocess
from pathlib import Path

REPO = Path("../").resolve()
RUNNERS = REPO / "dev/paper/runners"
RUNS = REPO / "dev/paper/runs"

def run(script_name: str):
    print(f"\n=== running {script_name} ===")
    out = subprocess.run(
        ["python", str(RUNNERS / script_name)],
        env={**os.environ},
        capture_output=True, text=True, timeout=900,
    )
    print(out.stdout[-2000:] if len(out.stdout) > 2000 else out.stdout)
    if out.returncode:
        print("STDERR:", out.stderr[-1000:])
    return out.returncode


## 1. The composite story (10 s)

Synthetic CeO2 ring frame → spatial dezinger + cosmic-ray rejection
→ empty-cell subtraction → polygon-bin integrate with σ propagation
→ PDF G(r) with σ → write DAT (PDFgetX3), FXYE (TOPAS), ESG
(MAUD/MILK), CSV (G(r) + σ).

**Talking point**: "Pixel σ → I(Q) σ → G(r) σ — propagated end-to-end
through corrections and the Fourier sine transform. pyFAI provides σ
at I(Q); v2 carries it the rest of the way to σ_G(r), which no
open-source PDF pipeline does today."


In [2]:
run("run_aps_meeting_demo.py")
print("outputs:", list((RUNS / "aps_meeting_demo").iterdir()))



=== running run_aps_meeting_demo.py ===


  dezinger: flagged 23532 pixels
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo

outputs: [PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/sample.fxye'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/REPORT.md'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/aps_iq.png'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/sample.esg'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/sample.gr'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/sample.dat'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/aps_gr.png'), PosixPath('/Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/aps_meeting_demo/

## 2. MIDAS-only feature: auto-detect bad pixels (~30 s)

Plant 200 cosmic-ray-like spikes; train LearnableMask jointly with a
sparsity prior; bad pixels naturally drop to 0.


In [3]:
run("run_learnable_mask_demo.py")



=== running run_learnable_mask_demo.py ===


  step   0  loss=3.860481e+05
  step  10  loss=3.449427e+05
  step  20  loss=2.970715e+05
  step  30  loss=2.363090e+05
  precision=0.000  recall=0.000  fp=0
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/learnable_mask_demo



0

## 3. MIDAS-only feature: auto-recover gain drift (~30 s)

Plant 5% Gaussian-smooth gain field; train LearnableGain to reverse it.
Recovered RMSE on the order of 1%.


In [4]:
run("run_learnable_gain_demo.py")



=== running run_learnable_gain_demo.py ===


  step   0  loss=1.216652e+03
  step  10  loss=7.904724e+01
  step  20  loss=5.755425e+01
  step  30  loss=1.033032e+01
  RMSE recovered vs truth: 1.0629e-02
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/learnable_gain_demo



0

## 4. Kernel comparison vs pyFAI (~60 s)

Hard / subpixel(K=2) / polygon kernels on the same synthetic CeO2;
pyFAI splitpixel(8×8) for cross-check (when pyFAI is installed).


In [5]:
run("run_pyfai_bakeoff.py")



=== running run_pyfai_bakeoff.py ===


Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/pyfai_bakeoff



0

## 5. σ chain pixel → MAUD (LANL Item 47, ~30 s)

Synthetic Ni standard with Poisson noise → MIDAS polygon σ → ESG with
σ in `_pd_proc_intensity_total_su`. Validates analytic σ against an
n=30 Monte Carlo bootstrap. Median rel-err is on the order of 0.1.


In [6]:
run("run_sigma_maud.py")



=== running run_sigma_maud.py ===


  median |σ_analytic - σ_bootstrap|/σ_bootstrap = 0.092
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/sigma_maud



0

## 6. HEDM grain-ODF vs E-WIMV (LANL Item 48, ~5 s)

Synthetic 2000-grain texture; volume-weighted (1,0,0) pole figure vs
E-WIMV-style smoothed analogue. KL and L2 distance reported. Real
LANL data plug-in via the same API once available.


In [7]:
run("run_hedm_ewimv_xval.py")



=== running run_hedm_ewimv_xval.py ===


  KL divergence: 1.0798e+00
  L2 distance:   3.0524e-02
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/hedm_ewimv_xval



0

## 7. Polarisation-plane refinement (~30 s)

Start from a wrong η_pol = 30°, fit it back to ≈ 0° (synchrotron
horizontal default) by minimising η-uniformity loss. Demonstrates
autograd-aware geometry refinement.


In [8]:
run("run_polarization_plane_refinement.py")



=== running run_polarization_plane_refinement.py ===


  converged η_pol = -4.393°
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/polarization_plane_refinement



0

## 8. SAXS workflow (Item 37, ~30 s)

Same framework, low-Q range. Solid-angle correction is the dominant
correction at small 2θ — 3.1× larger than polarisation, measured.


In [9]:
run("run_saxs_workflow.py")



=== running run_saxs_workflow.py ===


  SA effect / pol effect = 3.1x
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/saxs_workflow



0

## 9. Magnetic / resonant polarisation-resolved demo (~30 s)

Synthetic σ-pol vs π-pol Bragg intensity; ratio map shows magnetic
contrast.


In [10]:
run("run_magnetic_resonant_demo.py")



=== running run_magnetic_resonant_demo.py ===


  σ/π ratio mean: -763088879043247836744174796800.000
Wrote /Users/hsharma/opt/MIDAS/packages/midas_integrate_v2/dev/paper/runs/magnetic_resonant_demo



0

## 10. Real APS 1-ID Hydra CeO2 — multi-panel parity (~270 s)

Four panels × 2048² each, integrated through MILKMultiGeometryAdapter
into one σ-aware I(2θ) on a unified axis. **This is the LANL
Item 46 demo on real APS data.**

(Skip if you're tight on time — the talking point alone is enough.)


In [11]:
# Uncomment to run (~270 s on CPU)
# run("run_hydra_real_data.py")
print("data path: /Users/hsharma/Desktop/analysis/hydra/")
print("expected output: dev/paper/runs/hydra_real_data/hydra_combined.png")


data path: /Users/hsharma/Desktop/analysis/hydra/
expected output: dev/paper/runs/hydra_real_data/hydra_combined.png


## 11. APS-U operational throughput (~few min)

Eiger 9M-class detector (3262 × 3108) integrated through the polygon
kernel with σ propagation; throughput vs APS-U @ 100 Hz target.


In [12]:
# Uncomment to run (~5 min on CPU; produces fps figure)
# run("run_apsu_benchmark.py")
print("expected output: dev/paper/runs/apsu_benchmark/throughput.png")


expected output: dev/paper/runs/apsu_benchmark/throughput.png


## 12. Refresh PNGs for slide decks

Loads each runner's CSV/H5 outputs and writes PNGs into the
respective `dev/paper/runs/*/` directory.


In [13]:
run("plot_all_results.py")



=== running plot_all_results.py ===


plotting hydra_real_data
  wrote runs/hydra_real_data/hydra_combined.png
plotting hydra_sweep_real
  wrote runs/hydra_sweep_real/sweep.png
plotting sigma_maud
  wrote runs/sigma_maud/sigma_validation.png
plotting sigma_rietveld
plotting hedm_ewimv_xval
  wrote runs/hedm_ewimv_xval/hedm_vs_ewimv.png
plotting aps_meeting_demo
  wrote runs/aps_meeting_demo/aps_iq.png
  wrote runs/aps_meeting_demo/aps_gr.png
plotting pyfai_bakeoff
  wrote runs/pyfai_bakeoff/bakeoff.png
plotting learnable_mask_demo
  wrote runs/learnable_mask_demo/mask_recovery.png
plotting learnable_gain_demo
  wrote runs/learnable_gain_demo/gain_recovery.png
plotting polarization_plane_refinement
  wrote runs/polarization_plane_refinement/convergence.png
plotting saxs_workflow
  wrote runs/saxs_workflow/saxs_iq.png
plotting apsu_benchmark
  wrote runs/apsu_benchmark/throughput.png
plotting magnetic_resonant_demo
  wrote runs/magnetic_resonant_demo/polarization_ratio.png
plotting xrdct_demo
  found runs/xrdct_demo/figure.p

0

## Reference materials

- `dev/paper/HANDOFF.md` — full 48-item inventory with file pointers
- `dev/paper/meetings/INDEX.md` — meeting briefs index
- `dev/paper/meetings/DEMO_cheatsheet.md` — talking points per demo
- `dev/paper/meetings/slides/` — Marp slide decks (LANL / Wenqian / APS-mgmt / generic)
- `dev/paper/meetings/handouts/` — print-friendly 1-pagers
- `dev/aps_deployment_guide.md` — pan-APS recipes
